# Chapter 4: Bridging Vision & Language
## Projectors, Dynamic Tokens & Mixture-of-Experts

> **Papers covered:** LLaVA / LLaVA-1.5 (Liu et al. 2023), Flamingo (Alayrac et al. 2022), BLIP-2 (Li et al. 2023), DeepSeek-MoE (Dai et al. 2024)

---

### The Core Problem

A frozen Vision Transformer (ViT) outputs patch embeddings in $\mathbb{R}^{D_v}$.  
A frozen Large Language Model expects token embeddings in $\mathbb{R}^{D_l}$.  
These spaces are **dimensionally incompatible** and **semantically misaligned** — they were trained independently with different objectives on different data.

The **bridge architecture** is everything that lives between them.  It must solve two sub-problems:

1. **Dimensional alignment** — map $D_v \rightarrow D_l$  
2. **Semantic alignment** — ensure visual tokens are *interpretable* to the LLM

---

### Four Bridge Methods (Implemented From Scratch)

| # | Method | Paper | Output Tokens | Key Idea |
|---|--------|-------|--------------|----------|
| 2 | **MLP Projector** | LLaVA 1.5 | $N$ patches (no compression) | Simplest bridge — just align dimensions |
| 3 | **Perceiver Resampler** | Flamingo | $K \ll N$ (compression) | Learned queries compress visual tokens |
| 4 | **Q-Former** | BLIP-2 | $K$ queries | Pre-trained bridge w/ 3 objectives |
| 5 | **DeepStack** | Qwen-VL style | $N$ patches (multi-scale) | Fuse features from multiple ViT layers |

Plus a standalone **Mixture-of-Experts (MoE)** module (Section 6) to understand why frontier models like Qwen3-VL use sparse MoE backbones instead of dense transformers.


In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print(f"PyTorch: {torch.__version__}")


# 1) Setup: Shared Config & Synthetic Visual Features

## 1.1 Configuration

We use realistic dimensions throughout:
- **ViT-L/14** produces 1024-dim patch embeddings for a 224×224 image → 256 patches (16×16 grid) + 1 CLS token
- **LLaMA-7B** uses 4096-dim hidden states

All four bridges must accept `(batch_num, num_patches, vision_dim)` and produce `(batch_num, K, llm_dim)` where $K$ varies by method.

## 1.2 Synthetic Data

We simulate a **frozen ViT** by generating random tensors with the correct shapes.  
For `DeepStack`, we also simulate intermediate layer outputs (one per ViT block sampled).

> **Sample Input:**  
> `visual_feats.shape = (4, 257, 1024)` — batch of 4 images, 257 tokens (256 patches + CLS), 1024-dim  
>
> **Sample Multi-level Input:**  
> `[tensor(4,256,1024), tensor(4,256,1024), tensor(4,256,1024), tensor(4,256,1024)]` — 4 ViT layer outputs


In [ ]:
@dataclass
class BridgeConfig:
    vision_dim:    int = 1024   # ViT-L/14 output dimension
    llm_dim:       int = 4096   # LLaMA-7B hidden dimension
    num_patches:   int = 256    # 16×16 patch grid from 224×224 image
    num_queries:   int = 64     # Flamingo / BLIP-2 learned query count
    batch_num:     int = 4      # Mini-batch size for demos
    num_heads:     int = 8      # Multi-head attention heads
    num_levels:    int = 4      # ViT intermediate layers for DeepStack


cfg = BridgeConfig()


def make_visual_features(
    batch_num: int, num_patches: int, vision_dim: int
) -> torch.Tensor:
    """Simulate frozen ViT output: CLS token prepended to patch tokens."""
    # (batch_num, num_patches + 1, vision_dim)
    return torch.randn(batch_num, num_patches + 1, vision_dim, device=device)


def make_multilevel_features(
    batch_num: int, num_patches: int, vision_dim: int, num_levels: int
) -> List[torch.Tensor]:
    """Simulate outputs from multiple intermediate ViT transformer blocks."""
    # List of num_levels tensors, each: (batch_num, num_patches, vision_dim)
    return [
        torch.randn(batch_num, num_patches, vision_dim, device=device)
        for _ in range(num_levels)
    ]


# ── Verify shapes ──────────────────────────────────────────────────────────────
visual_feats = make_visual_features(cfg.batch_num, cfg.num_patches, cfg.vision_dim)
level_feats  = make_multilevel_features(cfg.batch_num, cfg.num_patches, cfg.vision_dim, cfg.num_levels)

print("Single-level visual features :", visual_feats.shape)
print(f"  → (batch_num={cfg.batch_num}, num_patches+1={cfg.num_patches+1}, vision_dim={cfg.vision_dim})")
print()
print(f"Multi-level features ({cfg.num_levels} levels):")
for i, f in enumerate(level_feats):
    print(f"  Level {i+1}: {f.shape}")


# 2) Method 1 — MLP Projector  *(The Winner)*

> **Paper:** LLaVA 1.5 — *"Improved Baselines with Visual Instruction Tuning"* (Liu et al. 2023)  
> **Used by:** LLaVA 1.5, Qwen-VL, InternVL, LLaVA-Next, Bunny, …

## 2.1 Intuition

LLaVA 1.0 used a **single linear layer** to bridge vision → language.  
LLaVA 1.5's key finding: adding **one more linear + GELU** dramatically improves performance.  
Adding a third layer brings no measurable gain.

Why does this work?  
- ViT and LLM feature spaces are both high-dimensional and roughly Gaussian  
- A learned affine + nonlinearity is sufficient to remap clusters without cross-attention  
- The simplicity means the projector trains end-to-end in the same pass as the rest of the model

**No compression**: all $N$ patch tokens pass through unchanged.  
The LLM sequence length is `(num_patches + num_text_tokens)`.

## 2.2 Sample Shapes

| Tensor | Shape |
|--------|-------|
| Input  | `(batch_num, num_patches, vision_dim)` = `(4, 257, 1024)` |
| Output | `(batch_num, num_patches, llm_dim)`   = `(4, 257, 4096)` |


In [ ]:
class MLPProjector(nn.Module):
    """
    Two-layer MLP projector — the de facto standard since LLaVA 1.5 (2023).

    Architecture:
        Linear(vision_dim → llm_dim) → GELU → Linear(llm_dim → llm_dim)

    Design decisions:
    - Hidden dim == output dim (not a bottleneck) because we want rich capacity
      for the nonlinear remap, not compression.
    - GELU matches the activation in most LLM FFN blocks, keeping gradient
      statistics consistent at the LLM input boundary.
    - Xavier uniform init avoids early saturation when vision_dim ≠ llm_dim.
    """

    def __init__(self, vision_dim: int, llm_dim: int) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
        self._init_weights()

    def _init_weights(self) -> None:
        for layer in self.mlp:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        # (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
        return self.mlp(visual_features)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ── Forward pass demo ──────────────────────────────────────────────────────────
mlp_proj = MLPProjector(cfg.vision_dim, cfg.llm_dim).to(device)
mlp_out  = mlp_proj(visual_feats)

print("MLPProjector")
print(f"  Input  shape : {visual_feats.shape}")
print(f"  Output shape : {mlp_out.shape}")
print(f"  Parameters   : {mlp_proj.count_parameters():,}")
print(f"  Token count  : {mlp_out.shape[1]} (no compression — all patches preserved)")


# 3) Method 2 — Cross-Attention Resampler  *(Flamingo)*

> **Paper:** Flamingo — *"A Visual Language Model for Few-Shot Learning"* (Alayrac et al. 2022, DeepMind)  
> **Also used by:** OpenFlamingo, IDEFICS, Otter

## 3.1 Intuition

Flamingo's insight: you don't need to hand *all* $N$ patch tokens to the LLM.  
Instead, learn $K$ **query vectors** that selectively *pull out* the most relevant information via cross-attention.

Think of each query as a "question" asked of the image:
- Query 1: *"What objects are in the foreground?"*
- Query 2: *"What is the background scene?"*
- Query 3: *"Are there any text or numbers?"*
- … (the network learns what questions to ask)

This **compresses** $N=256$ patch tokens down to $K=64$ query tokens — a 4× reduction in LLM sequence length, saving $O(K^2)$ attention compute inside the LLM.

## 3.2 Architecture

```
Learned queries  (K, D_l)  ←──────────────────────────────────────────────────────┐
                                                                                   │
Visual features  (N, D_v) ──→ kv_proj ──→ (N, D_l)  ──→  Cross-Attn  ──→ queries │
                                                   (queries = Q, visual = K,V)     │
                                                   ──→ FFN ──→ LayerNorm ──────────┘
```

## 3.3 Sample Shapes

| Tensor | Shape |
|--------|-------|
| Input  | `(batch_num, num_patches, vision_dim)` = `(4, 257, 1024)` |
| Output | `(batch_num, num_queries, llm_dim)`   = `(4, 64, 4096)` |


In [ ]:
class PerceiverLayer(nn.Module):
    """
    Single Perceiver block: cross-attention (queries attend to visual K/V) + FFN.
    Pre-norm (LayerNorm before attention) for training stability.
    """

    def __init__(self, llm_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            llm_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.ffn = nn.Sequential(
            nn.Linear(llm_dim, llm_dim * 4),
            nn.GELU(),
            nn.Linear(llm_dim * 4, llm_dim),
        )
        self.norm1   = nn.LayerNorm(llm_dim)
        self.norm2   = nn.LayerNorm(llm_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, queries: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        # Pre-norm cross-attention: queries are Q, visual features are K and V
        # queries: (batch_num, num_queries, llm_dim)
        # kv:      (batch_num, num_patches, llm_dim)
        residual = queries
        attn_out, _ = self.cross_attn(
            query=self.norm1(queries), key=kv, value=kv
        )
        queries = residual + self.dropout(attn_out)

        # FFN with residual connection
        # (batch_num, num_queries, llm_dim) → (batch_num, num_queries, llm_dim)
        residual = queries
        queries  = residual + self.dropout(self.ffn(self.norm2(queries)))
        return queries


class PerceiverResampler(nn.Module):
    """
    Cross-attention resampler from Flamingo (Alayrac et al. 2022).

    Key parameters:
        num_queries: K — the number of output tokens. Flamingo uses 64.
        num_layers:  depth of Perceiver stack. More layers = richer compression.

    The learned query vectors (self.queries) are the only persistent visual
    representation — they are *not* image-specific; they are task-level slots
    that get filled in differently per image during the forward pass.
    """

    def __init__(
        self,
        vision_dim:  int,
        llm_dim:     int,
        num_queries: int = 64,
        num_heads:   int = 8,
        num_layers:  int = 2,
        dropout:     float = 0.0,
    ) -> None:
        super().__init__()
        self.num_queries = num_queries

        # Persistent learned query slots — shared across the batch
        # (num_queries, llm_dim)
        self.queries = nn.Parameter(torch.randn(num_queries, llm_dim) * 0.02)

        # Project visual features from vision_dim → llm_dim (key/value space)
        self.kv_proj = nn.Linear(vision_dim, llm_dim)

        # Stack of Perceiver layers
        self.layers = nn.ModuleList(
            [PerceiverLayer(llm_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(llm_dim)

    def forward(self, visual_features: torch.Tensor) -> torch.Tensor:
        batch_num = visual_features.shape[0]

        # Project visual keys/values to llm_dim
        # (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
        kv = self.kv_proj(visual_features)

        # Broadcast learned queries over the batch dimension
        # (num_queries, llm_dim) → (batch_num, num_queries, llm_dim)
        queries = self.queries.unsqueeze(0).expand(batch_num, -1, -1)

        for layer in self.layers:
            # (batch_num, num_queries, llm_dim) → (batch_num, num_queries, llm_dim)
            queries = layer(queries, kv)

        # (batch_num, num_queries, llm_dim)
        return self.norm(queries)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ── Forward pass demo ──────────────────────────────────────────────────────────
resampler     = PerceiverResampler(cfg.vision_dim, cfg.llm_dim, cfg.num_queries, cfg.num_heads).to(device)
resampler_out = resampler(visual_feats)

print("PerceiverResampler")
print(f"  Input  shape : {visual_feats.shape}")
print(f"  Output shape : {resampler_out.shape}")
print(f"  Parameters   : {resampler.count_parameters():,}")
compression   = visual_feats.shape[1] / resampler_out.shape[1]
print(f"  Compression  : {visual_feats.shape[1]} → {resampler_out.shape[1]} tokens  ({compression:.1f}×)")


# 4) Method 3 — Q-Former  *(BLIP-2)*

> **Paper:** BLIP-2 — *"Bootstrapping Language-Image Pre-training"* (Li et al. 2023, Salesforce)  
> **Status:** Historically important; largely superseded by MLP projectors in practice.

## 4.1 Intuition

Q-Former goes further than the Perceiver by adding **self-attention among the queries** (not just cross-attention to visual features). This allows queries to communicate with each other and de-duplicate redundant information.

The architectural novelty of BLIP-2 was not the Q-Former itself, but its **three-objective pre-training**:

| Objective | What it learns |
|-----------|----------------|
| Image-Text Contrastive (ITC) | Align query representations with text embeddings |
| Image-grounded Text Matching (ITM) | Binary: does the text match the image? |
| Image-grounded Text Generation (ITG) | Generate causal text conditioned on queries |

This pre-training made Q-Former a *standalone transferable bridge* — you could freeze both ViT and LLM and only train Q-Former on image-text pairs.

## 4.2 Why It Was Superseded

- Adds ~200M parameters with a separate pre-training stage
- MLP projector (~20M params) trained end-to-end on instruction data matches or beats it
- Complex 3-objective training is hard to reproduce and tune

## 4.3 Sample Shapes

| Tensor | Shape |
|--------|-------|
| Input  | `(batch_num, num_patches, vision_dim)` = `(4, 257, 1024)` |
| Output | `(batch_num, num_queries=32, llm_dim)` = `(4, 32, 4096)` |


In [ ]:
class QFormerLayer(nn.Module):
    """
    Single Q-Former transformer layer.

    Difference from standard transformer layer:
      Standard: self-attn(all tokens) → cross-attn(all tokens, context) → FFN
      Q-Former:  self-attn(queries only)  → cross-attn(queries, visual) → FFN

    The queries never mix with text tokens at the layer level — only via the
    pre-training objectives (ITC/ITM/ITG) which are not implemented here.
    """

    def __init__(self, model_dim: int, num_heads: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.self_attn  = nn.MultiheadAttention(model_dim, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(model_dim, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.GELU(),
            nn.Linear(model_dim * 4, model_dim),
        )
        self.norm1   = nn.LayerNorm(model_dim)
        self.norm2   = nn.LayerNorm(model_dim)
        self.norm3   = nn.LayerNorm(model_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        queries:    torch.Tensor,
        visual_kv:  torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Self-attention: queries attend to each other (de-duplication)
        # (batch_num, num_queries, model_dim) → (batch_num, num_queries, model_dim)
        residual    = queries
        sa_out, _   = self.self_attn(self.norm1(queries), self.norm1(queries), self.norm1(queries))
        queries     = residual + self.dropout(sa_out)

        # Cross-attention: queries extract information from visual features
        # queries: (batch_num, num_queries, model_dim) — Q
        # visual_kv: (batch_num, num_patches, model_dim) — K and V
        residual      = queries
        ca_out, attn_w = self.cross_attn(
            query=self.norm2(queries), key=visual_kv, value=visual_kv
        )
        queries = residual + self.dropout(ca_out)

        # FFN
        # (batch_num, num_queries, model_dim) → (batch_num, num_queries, model_dim)
        residual = queries
        queries  = residual + self.dropout(self.ffn(self.norm3(queries)))

        return queries, attn_w


class QFormer(nn.Module):
    """
    Querying Transformer from BLIP-2 (Li et al. 2023).

    Simplified version: we implement the architecture but not the 3-objective
    pre-training (ITC / ITM / ITG), as that requires paired image-text data
    and is tangential to understanding the architectural contribution.

    Key parameters (matching BLIP-2 paper):
        num_queries = 32   (BLIP-2 uses 32)
        num_layers  = 6    (BLIP-2 uses a BERT_base-scale Q-Former: 12 layers)
        model_dim   = 768  (reduced here for demo)
    """

    def __init__(
        self,
        vision_dim:  int,
        model_dim:   int,
        llm_dim:     int,
        num_queries: int = 32,
        num_heads:   int = 8,
        num_layers:  int = 6,
        dropout:     float = 0.0,
    ) -> None:
        super().__init__()
        self.num_queries = num_queries

        # Learned query tokens — initialised like BERT [CLS]: truncated normal
        # (num_queries, model_dim) — shared, not image-specific
        self.query_tokens = nn.Parameter(torch.zeros(num_queries, model_dim))
        nn.init.trunc_normal_(self.query_tokens, std=0.02)

        # Project ViT features to Q-Former's internal space
        self.visual_proj = nn.Linear(vision_dim, model_dim)

        # Q-Former transformer blocks
        self.layers = nn.ModuleList(
            [QFormerLayer(model_dim, num_heads, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(model_dim)

        # Final projection: Q-Former dim → LLM dim
        self.output_proj = nn.Linear(model_dim, llm_dim)

    def forward(
        self, visual_features: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        batch_num = visual_features.shape[0]

        # Project visual features to Q-Former's model_dim (K and V space)
        # (batch_num, num_patches, vision_dim) → (batch_num, num_patches, model_dim)
        visual_kv = self.visual_proj(visual_features)

        # Broadcast learnable query tokens over batch
        # (num_queries, model_dim) → (batch_num, num_queries, model_dim)
        queries = self.query_tokens.unsqueeze(0).expand(batch_num, -1, -1)

        all_attn_weights: List[torch.Tensor] = []
        for layer in self.layers:
            # (batch_num, num_queries, model_dim) → (batch_num, num_queries, model_dim)
            queries, attn_w = layer(queries, visual_kv)
            all_attn_weights.append(attn_w)

        queries = self.norm(queries)

        # Upproject to LLM embedding space
        # (batch_num, num_queries, model_dim) → (batch_num, num_queries, llm_dim)
        output = self.output_proj(queries)

        return output, all_attn_weights

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ── Forward pass demo ──────────────────────────────────────────────────────────
QFORMER_MODEL_DIM = 768   # internal Q-Former dimension (BERT-base scale)
QFORMER_QUERIES   = 32

qformer     = QFormer(cfg.vision_dim, QFORMER_MODEL_DIM, cfg.llm_dim,
                      num_queries=QFORMER_QUERIES, num_heads=cfg.num_heads).to(device)
qformer_out, attn_maps = qformer(visual_feats)

print("Q-Former")
print(f"  Input  shape      : {visual_feats.shape}")
print(f"  Output shape      : {qformer_out.shape}")
print(f"  Parameters        : {qformer.count_parameters():,}")
compression = visual_feats.shape[1] / qformer_out.shape[1]
print(f"  Compression       : {visual_feats.shape[1]} → {qformer_out.shape[1]} tokens  ({compression:.1f}×)")
print(f"  Attention map[0]  : {attn_maps[0].shape}  (batch, num_queries, num_patches)")


# 5) Method 4 — DeepStack  *(Multi-Level Feature Fusion)*

> **Style:** Qwen3-VL, InternVL2, LLaVA-HR  
> **When to use:** OCR, chart understanding, dense grounding — tasks requiring fine-grained spatial detail.

## 5.1 Intuition

All three previous methods use **only the final ViT layer**.  
That's convenient but lossy: the final layer aggregates global context and discards local texture.

Different ViT layers encode different levels of visual abstraction:

```
Layer 1–3  :  Gabor-like edges, colour blobs
Layer 4–6  :  Textures, local patterns
Layer 7–9  :  Object parts, spatial relationships
Layer 10–12:  Semantic regions, global scene understanding
```

DeepStack samples features from $L$ intermediate layers, projects each to $D_l$, then **fuses** them.  
Two fusion strategies are implemented:

| Strategy | Formula | Notes |
|----------|---------|-------|
| Concat   | $\text{FFN}([f_1; f_2; \ldots; f_L])$ | Most expressive; more parameters |
| Gated Sum | $\sum_i \sigma(g_i) \cdot f_i$ | Interpretable; shows which layer matters |

## 5.2 Sample Shapes

| Tensor | Shape |
|--------|-------|
| Input  | List of `(batch_num, num_patches, vision_dim)` — one per level |
| Output | `(batch_num, num_patches, llm_dim)` = `(4, 256, 4096)` |


In [ ]:
class DeepStack(nn.Module):
    """
    Multi-level ViT feature fusion (Qwen3-VL / InternVL2 style).

    Unlike MLP / Perceiver / Q-Former which consume the final ViT layer only,
    DeepStack takes a *list* of intermediate layer outputs and fuses them.

    Two fusion modes:
        'concat'  — concatenate projections along the feature dim, then compress.
                    Most expressive; preferred when fine detail matters most.
        'sum'     — learned softmax-gated weighted sum.
                    Interpretable: inspect self.get_level_weights() to see which
                    ViT layer the model relies on most for a given task.
    """

    def __init__(
        self,
        vision_dim: int,
        llm_dim:    int,
        num_levels: int = 4,
        fusion:     str = "concat",   # "concat" | "sum"
    ) -> None:
        super().__init__()
        assert fusion in ("concat", "sum"), "fusion must be 'concat' or 'sum'"
        self.num_levels = num_levels
        self.fusion     = fusion

        # Per-level projection + normalisation
        # Each: (batch_num, num_patches, vision_dim) → (batch_num, num_patches, llm_dim)
        self.level_projs = nn.ModuleList([
            nn.Sequential(nn.Linear(vision_dim, llm_dim), nn.LayerNorm(llm_dim))
            for _ in range(num_levels)
        ])

        if fusion == "concat":
            # Compress concatenated multi-level features back to llm_dim
            # (batch_num, num_patches, llm_dim * num_levels) → (batch_num, num_patches, llm_dim)
            self.fusion_proj = nn.Sequential(
                nn.Linear(llm_dim * num_levels, llm_dim * 2),
                nn.GELU(),
                nn.Linear(llm_dim * 2, llm_dim),
            )
        else:
            # Scalar gate per level — softmax ensures they sum to 1
            # (num_levels,)
            self.level_gates = nn.Parameter(torch.ones(num_levels) / num_levels)

    def forward(self, level_features: List[torch.Tensor]) -> torch.Tensor:
        """
        level_features : List[Tensor] of length num_levels
                         Each tensor: (batch_num, num_patches, vision_dim)
        Returns        : (batch_num, num_patches, llm_dim)
        """
        assert len(level_features) == self.num_levels, (
            f"Expected {self.num_levels} levels, got {len(level_features)}"
        )

        # Project each ViT level independently
        # projected[i]: (batch_num, num_patches, llm_dim)
        projected = [proj(feat) for proj, feat in zip(self.level_projs, level_features)]

        if self.fusion == "concat":
            # Concatenate along feature dimension
            # (batch_num, num_patches, llm_dim * num_levels)
            fused = torch.cat(projected, dim=-1)
            # (batch_num, num_patches, llm_dim * num_levels) → (batch_num, num_patches, llm_dim)
            return self.fusion_proj(fused)

        # Gated sum fusion
        # gates: (num_levels,) after softmax
        gates = F.softmax(self.level_gates, dim=0)
        # Weighted sum: (batch_num, num_patches, llm_dim)
        return sum(g * feat for g, feat in zip(gates, projected))

    def get_level_weights(self) -> np.ndarray:
        """Return the normalised importance of each ViT level (gated-sum only)."""
        if self.fusion == "sum":
            return F.softmax(self.level_gates, dim=0).detach().cpu().numpy()
        return np.ones(self.num_levels) / self.num_levels

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ── Forward pass demo — both fusion modes ─────────────────────────────────────
deepstack_concat = DeepStack(cfg.vision_dim, cfg.llm_dim, cfg.num_levels, fusion="concat").to(device)
deepstack_sum    = DeepStack(cfg.vision_dim, cfg.llm_dim, cfg.num_levels, fusion="sum").to(device)

out_concat = deepstack_concat(level_feats)
out_sum    = deepstack_sum(level_feats)

print("DeepStack (concat fusion)")
print(f"  Input  : {cfg.num_levels} × {level_feats[0].shape}")
print(f"  Output : {out_concat.shape}")
print(f"  Params : {deepstack_concat.count_parameters():,}")

print()
print("DeepStack (gated-sum fusion)")
print(f"  Output : {out_sum.shape}")
print(f"  Params : {deepstack_sum.count_parameters():,}")
weights = deepstack_sum.get_level_weights()
for i, w in enumerate(weights):
    print(f"  Level {i+1} gate weight: {w:.4f}")

# ── Visualise gated-sum weights ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3))
level_labels = [f"ViT L{i+1}\n({'edges' if i==0 else 'texture' if i==1 else 'parts' if i==2 else 'scene'})"
                for i in range(cfg.num_levels)]
bars = ax.bar(level_labels, weights, color=["#4C72B0","#55A868","#C44E52","#8172B2"], alpha=0.85, width=0.5)
ax.set_ylabel("Softmax Gate Weight")
ax.set_title("DeepStack: Learned Level Importance (before training — uniform init)")
ax.set_ylim(0, 0.5)
for bar, w in zip(bars, weights):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f"{w:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


# 6) Mixture-of-Experts (MoE)

> **Papers:** Switch Transformer (Fedus et al. 2021), DeepSeek-MoE (Dai et al. 2024), Mixtral 8×7B (Jiang et al. 2024)  
> **Used by:** Qwen3-VL (235B total / 22B active), DeepSeek-V2, Mixtral, GPT-4 (rumoured)

## 6.1 Why MoE?

The efficiency argument in one equation:

```
Dense 7B model:    7B params active per token  →  7B FLOPs per token
MoE 235B model:   22B params active per token  → 22B FLOPs per token
                  (but 235B total capacity — ~10× more knowledge storage)
```

Each **Transformer FFN layer** is replaced by $N$ independent expert FFNs.  
A lightweight **router** selects the top-$K$ experts for each token.  
Only those $K$ experts run; the other $N-K$ are idle.

## 6.2 The Load Balancing Problem

Without regularisation, training collapses: the router always routes to 2–3 "favourite" experts.  
The rest never get gradients and remain random.

Solution: **Auxiliary load balancing loss** (Switch Transformer):

$$\mathcal{L}_{\text{aux}} = N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

Where $f_i$ = fraction of tokens assigned to expert $i$ (hard),  
and $P_i$ = mean routing probability for expert $i$ (soft, differentiable).  

Adding $\alpha \cdot \mathcal{L}_{\text{aux}}$ to the training loss encourages uniform utilisation.

## 6.3 Architecture Diagram

```
Token x  (model_dim,)
    │
    ├─► Router  Linear(model_dim → N)  ─► Softmax ─► Top-K select  ─► weights (K,), indices (K,)
    │
    ├─► Expert 0:  FFN(x)  ─── [selected or skipped]
    ├─► Expert 1:  FFN(x)  ─── [selected or skipped]
    ├─► ...
    └─► Expert N:  FFN(x)  ─── [selected or skipped]
    
Output = Σ_{k in TopK} weight_k × Expert_k(x)
```

## 6.4 Sample Shapes

| Tensor | Shape |
|--------|-------|
| Input  | `(batch_num, seq_len, model_dim)` |
| Output | `(batch_num, seq_len, model_dim)` (same — MoE is a drop-in FFN replacement) |
| Aux loss | scalar |


In [ ]:
class Expert(nn.Module):
    """
    Single FFN expert — identical architecture to a standard transformer FFN.
    All experts share the same shape but have independent weights.
    """

    def __init__(self, model_dim: int, expert_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(model_dim, expert_dim),
            nn.GELU(),
            nn.Linear(expert_dim, model_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (num_selected_tokens, model_dim) → (num_selected_tokens, model_dim)
        return self.net(x)


class TopKRouter(nn.Module):
    """
    Token routing network: maps token representations to (weights, expert indices).

    Design: a single weight-free-bias linear layer over num_experts logits.
    - Cheap: O(model_dim × num_experts) per token — negligible vs expert cost
    - Differentiable: gradients flow back to router via the softmax weights
    - No bias: prevents the router from collapsing to a constant independent of input

    Returns *three* things:
        weights      — (num_tokens, top_k)  softmax weights for selected experts
        indices      — (num_tokens, top_k)  which experts were selected
        router_probs — (num_tokens, num_experts)  full distribution for aux loss
    """

    def __init__(self, model_dim: int, num_experts: int, top_k: int) -> None:
        super().__init__()
        self.num_experts = num_experts
        self.top_k       = top_k
        self.gate        = nn.Linear(model_dim, num_experts, bias=False)

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        # Compute logits over all experts
        # (num_tokens, model_dim) → (num_tokens, num_experts)
        logits = self.gate(x)

        # Full softmax — used for load-balancing loss (differentiable)
        # (num_tokens, num_experts)
        router_probs = F.softmax(logits, dim=-1)

        # Select top-K expert indices and their unnormalised logits
        # (num_tokens, top_k), (num_tokens, top_k)
        top_k_logits, indices = torch.topk(logits, self.top_k, dim=-1)

        # Renormalise over only the selected K experts — mixture weights
        # (num_tokens, top_k)
        weights = F.softmax(top_k_logits, dim=-1)

        return weights, indices, router_probs


class MoELayer(nn.Module):
    """
    Mixture-of-Experts FFN layer — drop-in replacement for a dense transformer FFN.

    Compute comparison (K=2, N=8):
        Dense FFN:   1 × (2 · model_dim · expert_dim) FLOPs per token
        MoE layer:   2 × (2 · model_dim · expert_dim) FLOPs per token  (K=2 active)
                     but total parameter capacity = 8 × dense FFN

    ─────────────────────────────────────────────────────────────────────────────
    Dispatch loop strategy (used here):
        For each expert e, find the subset of tokens that selected e.
        Run expert e only on that subset.
        Accumulate weighted outputs.

    Alternative: expert-parallel with scatter/gather (used in production for GPU
    efficiency). The loop version here is pedagogically clearer.
    ─────────────────────────────────────────────────────────────────────────────
    """

    def __init__(
        self,
        model_dim:   int,
        num_experts: int = 8,
        top_k:       int = 2,
        expert_dim:  Optional[int] = None,
    ) -> None:
        super().__init__()
        self.model_dim   = model_dim
        self.num_experts = num_experts
        self.top_k       = top_k
        expert_dim       = expert_dim or model_dim * 4

        # N independent expert FFNs
        self.experts = nn.ModuleList(
            [Expert(model_dim, expert_dim) for _ in range(num_experts)]
        )
        # Lightweight token-to-expert router
        self.router = TopKRouter(model_dim, num_experts, top_k)

    def _aux_loss(
        self, router_probs: torch.Tensor, indices: torch.Tensor
    ) -> torch.Tensor:
        """
        Switch Transformer load-balancing loss.

        Encourages uniform expert utilisation by penalising the correlation
        between *how often* an expert is selected (hard count) and *how likely*
        the router assigns probability to it (soft mean).

        L_aux = num_experts × Σ_i  fraction_i × mean_prob_i

        At perfect balance: fraction_i = 1/N, so L_aux = 1/N × Σ mean_prob_i = 1.
        Imbalanced routing increases L_aux above 1, providing a penalty signal.
        """
        num_tokens = router_probs.shape[0]

        # One-hot encode selected experts → fraction of tokens per expert
        # (num_tokens, top_k) → (num_tokens, top_k, num_experts) → (num_experts,)
        expert_mask = F.one_hot(indices, num_classes=self.num_experts).float()
        fraction    = expert_mask.sum(dim=(0, 1)) / (num_tokens * self.top_k)

        # Differentiable mean routing probability per expert
        # (num_experts,)
        mean_prob = router_probs.mean(dim=0)

        # Scalar load-balancing loss
        return self.num_experts * (fraction * mean_prob).sum()

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        batch_num, seq_len, model_dim = x.shape

        # Flatten batch × sequence for routing
        # (batch_num, seq_len, model_dim) → (num_tokens, model_dim)
        x_flat     = x.reshape(-1, model_dim)
        num_tokens = x_flat.shape[0]

        # Route tokens: get combination weights and expert indices
        # weights: (num_tokens, top_k)  |  indices: (num_tokens, top_k)
        weights, indices, router_probs = self.router(x_flat)

        # Accumulator for expert outputs
        # (num_tokens, model_dim)
        output = torch.zeros_like(x_flat)

        # Dispatch: loop over top-k slots, then over experts
        for k in range(self.top_k):
            expert_idx    = indices[:, k]         # (num_tokens,)
            expert_weight = weights[:, k, None]   # (num_tokens, 1)

            for e, expert in enumerate(self.experts):
                # Identify tokens routed to expert e at this k-slot
                token_mask = expert_idx == e     # (num_tokens,) bool
                if not token_mask.any():
                    continue

                # Run selected tokens through expert e
                # (num_selected, model_dim) → (num_selected, model_dim)
                expert_out        = expert(x_flat[token_mask])
                output[token_mask] += expert_weight[token_mask] * expert_out

        aux_loss = self._aux_loss(router_probs, indices)

        # Reshape output back to original batch structure
        # (num_tokens, model_dim) → (batch_num, seq_len, model_dim)
        return output.reshape(batch_num, seq_len, model_dim), aux_loss

    def get_routing_stats(self, x: torch.Tensor) -> Dict:
        """Analyse expert utilisation for a given input batch (no gradient)."""
        with torch.no_grad():
            x_flat             = x.reshape(-1, self.model_dim)
            _, indices, probs  = self.router(x_flat)
            expert_counts      = torch.zeros(self.num_experts)
            for e in range(self.num_experts):
                expert_counts[e] = (indices == e).sum().item()
            return {
                "expert_counts": expert_counts.numpy(),
                "mean_prob":     probs.mean(dim=0).cpu().numpy(),
                "router_entropy": -(probs * torch.log(probs + 1e-8)).sum(-1).mean().item(),
            }

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters())


# ── Forward pass demo ──────────────────────────────────────────────────────────
MOE_SEQ_LEN  = 16
MOE_DIM      = 256
MOE_EXPERTS  = 8
MOE_TOPK     = 2

moe_layer = MoELayer(MOE_DIM, num_experts=MOE_EXPERTS, top_k=MOE_TOPK).to(device)
x_moe     = torch.randn(cfg.batch_num, MOE_SEQ_LEN, MOE_DIM, device=device)
moe_out, aux = moe_layer(x_moe)

print("MoELayer")
print(f"  Input  shape   : {x_moe.shape}")
print(f"  Output shape   : {moe_out.shape}")
print(f"  Aux loss       : {aux.item():.4f}  (target ≈ 1.0 at perfect balance)")
print(f"  Parameters     : {moe_layer.count_parameters():,}")

# ── Routing statistics ─────────────────────────────────────────────────────────
stats = moe_layer.get_routing_stats(x_moe)
print()
print("Expert utilisation (random init — expect near-uniform):")
for e in range(MOE_EXPERTS):
    bar = "█" * int(stats["expert_counts"][e])
    print(f"  Expert {e}: {int(stats['expert_counts'][e]):>4} tokens  {bar}")
print(f"  Router entropy: {stats['router_entropy']:.4f}")


In [ ]:
# ── Visualise routing distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

expert_labels = [f"E{i}" for i in range(MOE_EXPERTS)]

axes[0].bar(expert_labels, stats["expert_counts"], color="#4C72B0", alpha=0.85)
axes[0].axhline(
    y=(cfg.batch_num * MOE_SEQ_LEN * MOE_TOPK) / MOE_EXPERTS,
    color="red", linestyle="--", linewidth=1.5, label="Perfect balance"
)
axes[0].set_title("Token Count per Expert")
axes[0].set_ylabel("Tokens routed")
axes[0].legend()

axes[1].bar(expert_labels, stats["mean_prob"], color="#55A868", alpha=0.85)
axes[1].axhline(y=1.0 / MOE_EXPERTS, color="red", linestyle="--", linewidth=1.5,
                label="Uniform (1/N)")
axes[1].set_title("Mean Router Probability per Expert")
axes[1].set_ylabel("Mean probability")
axes[1].legend()

plt.suptitle(
    f"MoE Routing Stats  |  {MOE_EXPERTS} experts, top-{MOE_TOPK}  |  "
    f"Random init → near-uniform before training",
    fontsize=11
)
plt.tight_layout()
plt.show()

# ── Simulate expert collapse (what happens without aux loss) ──────────────────
print("\nSimulating expert collapse (biased router):")
with torch.no_grad():
    biased_gate           = moe_layer.router.gate.weight.clone()
    biased_gate[0]       *= 20     # make expert 0 always win
    moe_layer.router.gate.weight.copy_(biased_gate)
    collapsed_stats = moe_layer.get_routing_stats(x_moe)
    print("  Expert counts after bias injection:")
    for e in range(MOE_EXPERTS):
        bar = "█" * int(collapsed_stats["expert_counts"][e])
        print(f"    Expert {e}: {int(collapsed_stats['expert_counts'][e]):>4}  {bar}")
    print(f"  Router entropy (collapsed): {collapsed_stats['router_entropy']:.4f}  ← much lower")

# Restore original weights
nn.init.xavier_uniform_(moe_layer.router.gate.weight)


# 7) Comparative Analysis

## 7.1 What We're Measuring

We compare all four bridge architectures on three axes:

1. **Parameter count** — how many trainable weights does the bridge add?
2. **Output token count** — how many tokens does the LLM receive from vision?
3. **Compute budget proxy** — token count × llm_dim² (the dominant cost in LLM self-attention)

## 7.2 Design Trade-off Space

```
                HIGH DETAIL
                    │
    DeepStack ──────┤──────────────────── MLP Projector
    (multi-scale)   │                   (simple, fast)
                    │
    ────────────────┼──────────── COMPRESSION
                    │
    Q-Former ───────┤──────────── Perceiver Resampler
    (complex,       │            (elegant, moderate)
     pre-trained)   │
                    │
                LOW DETAIL
```

## 7.3 Conclusion

> **Why the field converged on MLP + MoE backbone:**
>
> - MLP projector is 10–100× simpler than Q-Former/Perceiver, trains end-to-end, and performs comparably on most benchmarks when paired with strong instruction-tuning data.
> - DeepStack is worth the overhead only when fine-grained detail is the bottleneck (OCR, charts, grounding).
> - MoE backbone provides free capacity scaling: 10× more parameters at 2–3× more compute — the most efficient way to scale a multimodal model.


In [ ]:
def build_all_bridges(cfg: BridgeConfig) -> Dict:
    """Instantiate all four bridge architectures and collect metadata."""
    bridges = {}

    # MLP Projector
    bridges["MLP Projector\n(LLaVA 1.5)"] = {
        "model":          MLPProjector(cfg.vision_dim, cfg.llm_dim).to(device),
        "output_tokens":  cfg.num_patches + 1,
        "description":    "Simple; no compression",
        "paper":          "LLaVA 1.5",
    }

    # Perceiver Resampler
    bridges["Perceiver\nResampler\n(Flamingo)"] = {
        "model":          PerceiverResampler(cfg.vision_dim, cfg.llm_dim,
                              cfg.num_queries, cfg.num_heads).to(device),
        "output_tokens":  cfg.num_queries,
        "description":    "4× compression via cross-attention",
        "paper":          "Flamingo",
    }

    # Q-Former
    bridges["Q-Former\n(BLIP-2)"] = {
        "model":          QFormer(cfg.vision_dim, 768, cfg.llm_dim,
                              num_queries=32, num_heads=cfg.num_heads).to(device),
        "output_tokens":  32,
        "description":    "8× compression; pre-trained",
        "paper":          "BLIP-2",
    }

    # DeepStack
    bridges["DeepStack\n(Qwen-VL)"] = {
        "model":          DeepStack(cfg.vision_dim, cfg.llm_dim, cfg.num_levels).to(device),
        "output_tokens":  cfg.num_patches,
        "description":    "Multi-scale; no compression",
        "paper":          "Qwen-VL",
    }

    for name, entry in bridges.items():
        entry["params"] = entry["model"].count_parameters()
        # Proxy for LLM attention cost: O(output_tokens² × llm_dim)
        entry["attn_cost_proxy"] = entry["output_tokens"] ** 2 * cfg.llm_dim

    return bridges


bridges = build_all_bridges(cfg)

print(f"{'Method':<28} {'Params':>12} {'Output tokens':>14} {'Compression':>12}")
print("─" * 72)
for name, entry in bridges.items():
    n      = name.replace("\n", " ")
    ratio  = (cfg.num_patches + 1) / entry["output_tokens"]
    print(f"{n:<28} {entry['params']:>12,} {entry['output_tokens']:>14} {ratio:>11.1f}×")


In [ ]:
# ── Side-by-side bar charts ────────────────────────────────────────────────────
labels      = [n.replace("\n", "\n") for n in bridges.keys()]
params      = [b["params"] / 1e6 for b in bridges.values()]
out_tokens  = [b["output_tokens"] for b in bridges.values()]
attn_costs  = [b["attn_cost_proxy"] / 1e9 for b in bridges.values()]
colors      = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

fig = plt.figure(figsize=(16, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# Parameters
ax1 = fig.add_subplot(gs[0])
bars = ax1.bar(labels, params, color=colors, alpha=0.85, width=0.5)
ax1.set_title("Bridge Parameters (M)", fontweight="bold")
ax1.set_ylabel("Parameters (millions)")
for bar, v in zip(bars, params):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{v:.1f}M", ha="center", fontsize=8)

# Output token count
ax2 = fig.add_subplot(gs[1])
bars2 = ax2.bar(labels, out_tokens, color=colors, alpha=0.85, width=0.5)
ax2.set_title("Output Tokens (LLM input length)", fontweight="bold")
ax2.set_ylabel("Number of visual tokens")
ax2.axhline(cfg.num_queries, color="gray", linestyle=":", linewidth=1, label="Flamingo K=64")
for bar, v in zip(bars2, out_tokens):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             str(v), ha="center", fontsize=8)

# Attention cost proxy
ax3 = fig.add_subplot(gs[2])
bars3 = ax3.bar(labels, attn_costs, color=colors, alpha=0.85, width=0.5)
ax3.set_title("LLM Attention Cost Proxy\n(tokens² × llm_dim, ×10⁹)", fontweight="bold")
ax3.set_ylabel("Relative cost (×10⁹)")
for bar, v in zip(bars3, attn_costs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f"{v:.2f}", ha="center", fontsize=8)

plt.suptitle(
    f"Bridge Architecture Comparison  |  vision_dim={cfg.vision_dim}, llm_dim={cfg.llm_dim}",
    fontsize=12, fontweight="bold", y=1.01
)
plt.savefig("bridge_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved: bridge_comparison.png")


# 8) Summary & Design Guidelines

## 8.1 Decision Framework

```
Do you need token compression?
    YES → Perceiver Resampler  (simple, single pre-training stage)
          Q-Former             (if you have resources for 3-obj pre-training)
    NO  → Does the task require fine-grained spatial detail?  (OCR, charts, grounding)
              YES → DeepStack   (fuse multiple ViT layers)
              NO  → MLP Projector  (default choice — simplest, fastest, often best)
```

## 8.2 MoE Backbone Decision

```
Model scale < 7B dense equivalent?   →  Dense transformer (simpler to train)
Model scale ≥ 13B?                   →  MoE backbone strongly recommended
                                         22B active / 235B total is now standard
```

## 8.3 The Convergence Story (2022 → 2024)

| Year | Architecture | Key lesson |
|------|-------------|------------|
| 2022 | Flamingo: Perceiver Resampler | Cross-attention resampling works; compression viable |
| 2023 | BLIP-2: Q-Former | Pre-trained bridges transfer well; but complex |
| 2023 | LLaVA 1.5: MLP | With good data, simpler beats complex |
| 2024 | Qwen3-VL: DeepStack + MoE | Multi-scale detail + sparse compute = frontier |

## 8.4 What You Built

- ✅ **MLPProjector** — 2-layer MLP, Xavier init, GELU, shape-annotated
- ✅ **PerceiverResampler** — learned queries + multi-layer cross-attention + FFN
- ✅ **QFormer** — alternating self-attention + cross-attention, output projection
- ✅ **DeepStack** — per-level projection, concat and gated-sum fusion modes
- ✅ **MoELayer** — TopKRouter + N experts + dispatch loop + load balancing loss
- ✅ **Comparative analysis** — parameter counts, token counts, attention cost proxy
